only accepting full double barrel scientific names and variants no only genus or only species names
## **Bulk plant normalizer**
#### **This plant normalizer should do the following:**
- take in and format the wcvp
- take in a file retured from the checklist bank that has been deduplicated on the origional name
- keep only the GBIF name, ID, accepted name, accepted ID
- remove non-plants
- remove any genus or species only names (need both genus and species minimum)
- adding a cleaned name and matching the datasets based on that 
- fuzzy matching on the cleaned names using the fuzz.ratio scorer
- linking back to accepted ID
- collapsing into unified main ID with all else going to synonyms

In [ ]:
wcvp_file = ''
new_checklisted_file = ''
staging_table= ''
wand_plants = ''
error_output_path = ''
output_path = ''
report_path = ''
#////////////////////////////////////////////////////////imports
import pandas as pd
from rapidfuzz import process
#////////////////////////////////////////////////////////functions
def name_cleaner(name,id):
    global errors
    if pd.isna(name):
        err = [id, 'Name cleaning error', 'There was no name for this entry']
        errors.loc[len(errors)] = err
        return pd.NA
    if 'sp' in name or 'spp' in name:
        err = [id, 'Name error', f'{name} contains "sp" or "spp" and was therefore excluded']
        errors.loc[len(errors)] = err
        return pd.NA
    name = name.strip().lower().replace('-','').split()
    name = ' '.join(name)    
    return name

def is_complete_name(name,id):
    global errors
    og_name = name
    if pd.isna(name) or name == '' or not name.strip():
        err = [id,'Incomplete name','There was no name for this entry']
        errors.loc[len(errors)] = err
        return False
    else:
        name = name.strip()
        if len(name.split()) < 2:
            return True
        else:
            err = [id,'Incomplete name',f'{og_name} is not a valid name (likely was only genus or species)']
            errors.loc[len(errors)] = err
            return False

def fuzzy_match(og_name, og_id, name_id_dict):
    cutoff = 95
    result = process.extractOne(og_name,name_id_dict.keys())
    if result is None:
        err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to none of the wcvp']
        errors.loc[len(errors)] = err
        return pd.NA
    match, chance, _ = result
    if chance < cutoff:
        err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to {match} at {chance}% chance']
        errors.loc[len(errors)] = err
        return pd.NA
    else:
        return name_id_dict.get(match)
    

#////////////////////////////////////////////////////////data
                                #////////////////////////load data
print('Loading data...', flush = True)
error_messages =''
wcvp = pd.read_csv(wcvp_file, sep='|')
new = pd.read_csv(new_checklisted_file)
errors = pd.DataFrame(columns=['id','error_stage','exact_error'])
                                #////////////////////////format wcvp
print('Formatting wcvp...', flush = True)
wcvp = wcvp.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'})
wcvp.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)

wcvp_name_check = wcvp[['scientific_name','taxonid']]
wcvp_name_check['clean_name'] = wcvp_name_check['scientific_name'].apply(name_cleaner)
wcvp_name_check_dic = {row.clean_name: row.taxonid for row in wcvp_name_check.itertuples()}
#////////////////////////////////////////////////////////normalize new plants
                                #////////////////////////drop unnecessary columns
new = new[['original_id','original_scientificName','matchType','matchIssues','ID','sceintificName','acceptedID','acceptedScientificName','kingdom']]
                                #////////////////////////removing non-plants
print('Removing non-plants from gbif match...', flush = True)
is_plant = new['kingdom'].eq('Plantae').fillna(False)
new_plants = new.loc[is_plant].copy()
non_plants = new.loc[~is_plant].copy()
no_kingdoms = non_plants.loc[non_plants['kingdom'].isna()].copy()
no_kingdoms_err = pd.DataFrame({'id': no_kingdoms['origional_id'],'error_stage': 'No gbif kingdom', 'exact_error': (no_kingdoms['original_scientificName'] + ' entry excluded there was no kingdom')})
errors.append(no_kingdoms_err)
non_plants = non_plants.loc[non_plants['kingdom'].notna()].copy()
non_plant_errors  = pd.DataFrame({'id': non_plants['origional_id'],'error_stage': 'GBIF kingdom was not plantae', 'exact_error': (non_plants['original_scientificName'] + ' not recognised by gbif as a plant')})
errors = errors.append(non_plant_errors)
                                #////////////////////////removing incomplete names
print('Removing incomplete names...', flush = True)
new_plants['name_complete'] = new_plants.apply(lambda row: is_complete_name(row['original_scientificName'],row['origional_id']),axis = 1)
new_plants = new_plants.loc[new_plants['name_complete']].copy()
                                #////////////////////////adding clean name for direct matching
new_plants['clean_name'] = new_plants.apply(lambda row: name_cleaner(row['original_scientificName'],row['origional_id']),axis = 1)
new_plants.dropna(subset=['clean_name'],inplace=True)
                                #////////////////////////checking for duplicates on clean name and removing if necessary
if new_plants.duplicated(subset='clean_name').any():
   vfbcxbxcvb error_messages = error_messages + 'Once the names were cleaned for direct original name wcvp matching some of the cleaned names were duplicated the script keeps the first and removes the rest of the duplicates. Please ensure the original file and the checklist return file are deduplicated on the plant name. I this was done then the actual cleaning process leads to duplicates of names by removing "-" and whitespace as well as comparing only lowercase. This means redundancy was likely removed. The removed entries are in the error file. \n'
   vfbcxbxcvb duplicated = new_plants.loc[new.duplicated(subset='clean_name',keep=False)].copy()
   vfbcxbxcvb duplicated_not_removed = duplicated.loc[duplicated.duplicated(subset='clean_name',keep='first')]['clean_name'].to_list()
   vfbcxbxcvb duplicated = duplicated.loc[~duplicated['clean_name'].isin(duplicated_not_removed)].copy()
   vfbcxbxcvb err = [[row.original_id,'Duplication',f'{row.original_name} cleaned to {row.clean_name} was duplicated after cleaning'] for row in duplicated.itertuples()]
   vfbcxbxcvb errors.loc[len(errors)] = err
   vfbcxbxcvb new_plants = new_plants.drop_duplicates(subset='clean_name',keep='first')
                                #////////////////////////seperating new plants from plants in the db already
                                #////////////////////////matching cleaned names
new_plants = new_plants.merge(wcvp_name_check, how='left',on='clean_name', validate='many_to_one')
                                #////////////////////////fuzzy matching what is left
unmatched = new.loc[new['taxonid'].isna()].copy()
new_plants['taxonid_fuzzy'] = unmatched.apply(lambda row: fuzzy_match(row.clean_name, row.original_id, wcvp_name_check_dic),axis=1)
                                #////////////////////////recording match numbers
new_plants['match_type'] = pd.NA
new_plants.loc[new_plants['taxonid'].notna(),'match_type'] = 'direct_match'
new_plants.loc[new_plants['taxonid_fuzzy'].notna(),'match_type'] = 'fuzzy match' 
                                #////////////////////////removing all unmatched
#there is no need for an error handler here to add the dropped as they were added already during fuzzy match failure
new_plants.dropna(subset='match_type',inplace=True)
                                #////////////////////////using taxon id to get no accepted info

                                #////////////////////////using accepted usage name id to get accepted info

                                #////////////////////////putting row non accepted into gbif and powo synonyms

                                #////////////////////////taking every accepted id that is taxon id and adding synonyms from where it is listed as accepted usage name id  

                                #////////////////////////formatting for DB with plant_id and correct columns

                                #remove entities that are weird matches
#////////////////////////////////////////////////////////plants already in db

#////////////////////////////////////////////////////////report
stats = f"""The number matched directly: {direct_matches:,} \n
            The number fuzzy matched: {fuzzy_matches:,}"""

item = error_messages + '\n' + stats
with open(report_path, 'w') as f:
    f.write(item)

#////////////////////////////////////////////////////////output